In [28]:
# !python -m spacy download it_core_news_lg
# !pip install stanza

In [29]:
import time
import numpy as np
import json

import spacy
import stanza

import pandas as pd

stanza.download("it")  # once

2025-10-16 11:19:07 INFO: Downloaded file to /Users/stevie/stanza_resources/resources.json
2025-10-16 11:19:07 INFO: Downloading default packages for language: it (Italian) ...
2025-10-16 11:19:08 INFO: File exists: /Users/stevie/stanza_resources/it/default.zip
2025-10-16 11:19:09 INFO: Finished downloading models and saved to /Users/stevie/stanza_resources


In [30]:

class SpacyTokenizer:
    def __init__(self):
        self.nlp = spacy.load("it_core_news_lg")

    def tokenize(self, text):
        doc = self.nlp(text)
        return [
            {
                "text": token.text,
                "lemma": token.lemma_,
                "pos": token.pos_,
                "tag": token.tag_,
                "dep": token.dep_,
                "shape": token.shape_,
                "is_alpha": token.is_alpha,
                "is_stop": token.is_stop,
                "vector": token.vector if np.any(token.vector) else None,
            } 
            for token in doc
        ]

class StanzaTokenizer:
    def __init__(self):
        self.nlp = stanza.Pipeline("it", processors="tokenize,pos,lemma,depparse")

    def tokenize(self, text):
        doc = self.nlp(text)
        tokens = []
        for sentence in doc.sentences:
            for token in sentence.tokens:
                word = token.words[0]
                tokens.append({
                    "text": token.text,
                    "lemma": word.lemma,
                    "pos": word.upos,
                    "xpos": word.xpos,
                    "head": word.head,
                    "deprel": word.deprel,
                    "vector": None,  # Stanza does not provide word vectors
                })
        return tokens

In [31]:
spacy_tokenizer = SpacyTokenizer()
stanza_tokenizer = StanzaTokenizer()

2025-10-16 11:19:10 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-10-16 11:19:10 INFO: Downloaded file to /Users/stevie/stanza_resources/resources.json
2025-10-16 11:19:10 WARNING: Language it package default expects mwt, which has been added
2025-10-16 11:19:11 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2025-10-16 11:19:11 INFO: Using device: cpu
2025-10-16 11:19:11 INFO: Loading: tokenize
2025-10-16 11:19:11 INFO: Loading: mwt
2025-10-16 11:19:11 INFO: Loading: pos
2025-10-16 11:19:11 INFO: Loading: lemma
2025-10-16 11:19:12 INFO: Loading: depparse
2025-10-16 11:19:12 INFO: Done loading 

In [32]:
pos_test_cases = json.load(open("testcases/test_pos.json", "r"))
def run_pos_tests(tokenizer):
    pass_count = 0
    fail_count = 0
    time_start = time.time()
    for test_case in pos_test_cases:
        key_word = test_case['key_word']

        for sentence in test_case['sentences']:
            text = sentence['text']
            text_en = sentence['text_en']
            expected_pos = sentence['expected']['pos']
            expected_lemma = sentence['expected']['lemma']

            tokens = tokenizer.tokenize(text)

            # Track all occurrences of the key word and any matches to expected
            occurrences = []
            matched = False

            for w in tokens:
                if w['text'].lower() == key_word.lower():
                    occ = {"pos": w['pos'], "lemma": w['lemma']}
                    occurrences.append(occ)
                    # if w.upos == expected_pos and w.lemma == expected_lemma:
                    if w['pos'] == expected_pos:
                        matched = True

            if matched:
                # Only a concise PASS line when expectations are met
                # print(f"{key_word}: PASS")
                pass_count += 1
            else:
                # Detailed output only on failure
                print(f"{key_word}: FAIL")
                print(f"Sentence: {text} ({text_en})")
                if not occurrences:
                    print(f"  Word '{key_word}' not found in the sentence.")
                else:
                    print(f"  Expected -> POS: {expected_pos}, Lemma: {expected_lemma}")
                    print("  Found ->")
                    for i, occ in enumerate(occurrences, 1):
                        print(f"    Occurrence {i}: POS={occ['pos']}, Lemma={occ['lemma']}")
                fail_count += 1
    time_end = time.time()
    print(f"\nTotal PASS: {pass_count}, Total FAIL: {fail_count}")
    print(f"Time taken: {time_end - time_start:.2f} seconds\n")

In [33]:
print("Testing SpaCy Tokenizer")
run_pos_tests(spacy_tokenizer)

Testing SpaCy Tokenizer
volo: FAIL
Sentence: Oggi volo a Milano. (Today I fly to Milan.)
  Expected -> POS: VERB, Lemma: volare
  Found ->
    Occurrence 1: POS=NOUN, Lemma=volo
sinistra: FAIL
Sentence: La vecchia casa era sinistra. (The old house was sinister.)
  Expected -> POS: ADJ, Lemma: sinistro
  Found ->
    Occurrence 1: POS=NOUN, Lemma=sinistra
pranzo: FAIL
Sentence: Domani pranzo con Luca. (Tomorrow I have lunch with Luca.)
  Expected -> POS: VERB, Lemma: pranzare
  Found ->
    Occurrence 1: POS=NOUN, Lemma=pranzo
sogno: FAIL
Sentence: Sogno spesso di volare. (I often dream of flying.)
  Expected -> POS: VERB, Lemma: sognare
  Found ->
    Occurrence 1: POS=NOUN, Lemma=sogno
cambio: FAIL
Sentence: Domani cambio lavoro. (Tomorrow I change jobs.)
  Expected -> POS: VERB, Lemma: cambiare
  Found ->
    Occurrence 1: POS=NOUN, Lemma=cambio
diritto: FAIL
Sentence: Studio diritto all'università. (I study law at university.)
  Expected -> POS: NOUN, Lemma: diritto
  Found ->
    O

In [34]:
print("Testing Stanza Tokenizer")
run_pos_tests(stanza_tokenizer)

Testing Stanza Tokenizer
piano: FAIL
Sentence: Parla piano, per favore. (Speak softly, please.)
  Expected -> POS: ADV, Lemma: piano
  Found ->
    Occurrence 1: POS=NOUN, Lemma=piano
pranzo: FAIL
Sentence: Domani pranzo con Luca. (Tomorrow I have lunch with Luca.)
  Expected -> POS: VERB, Lemma: pranzare
  Found ->
    Occurrence 1: POS=NOUN, Lemma=pranzo
cambio: FAIL
Sentence: Domani cambio lavoro. (Tomorrow I change jobs.)
  Expected -> POS: VERB, Lemma: cambiare
  Found ->
    Occurrence 1: POS=NOUN, Lemma=cambio
diritto: FAIL
Sentence: È un palo diritto. (It's a straight pole.)
  Expected -> POS: ADJ, Lemma: diritto
  Found ->
    Occurrence 1: POS=NOUN, Lemma=diritto

Total PASS: 20, Total FAIL: 4
Time taken: 1.22 seconds



In [35]:
lemma_test_cases = json.load(open("testcases/test_lemma.json", "r"))

In [36]:
data = {
    'word': [],
    'expected_lemma': [],
    'expected_pos': [],
    'sentence': [],
    'sentence_en': [],
}
for tc in lemma_test_cases['test_cases']:
    kw = tc['key_word']
    for sent in tc['sentences']:
        data['word'].append(kw)
        data['expected_lemma'].append(sent['expected']['lemma'])
        data['expected_pos'].append(sent['expected']['pos'])
        data['sentence'].append(sent['text'])
        data['sentence_en'].append(sent['text_en'])
df = pd.DataFrame(data)

In [37]:
tokenizer = stanza_tokenizer
for i, row in df.iterrows():
    sent = row['sentence']
    tokens = stanza_tokenizer.tokenize(sent)
    for t in tokens:
        if t['text'].lower() == row['word'].lower():
            df.loc[i, 'predicted_lemma'] = t['lemma']
            df.loc[i, 'predicted_pos'] = t['pos']

In [ ]:
df.sort_values('expected_lemma', inplace=True)
df.to_csv('temp.csv', index=False)

In [41]:
tokenizer = stanza_tokenizer
sentences = [
    'Con più ferie volereste più spesso?',
    'Se ci fosse meno vento, volereste anche oggi.',
    'A che ora volereste domani?',
    'Volereste davvero attraversare le Alpi di notte?',
    'In quale compagnia volereste per risparmiare?',
    'Non volereste con un aereo così vecchio, vero?',
    'Mi confermate che volereste in economy?',
    'Se aveste il passaporto, volereste domani stesso.',
    'Volereste anche fare scalo a Roma?',
    'Dove volereste con quei punti?',
    'In estate volereste quasi ogni weekend.',
    'Con quel budget volereste in economy.',
]
for sentence in sentences:
    print(sentence)
    tokens = tokenizer.tokenize(sentence)
    for t in tokens:
        if t['text'].lower() == 'volereste':
            print(t['text'], t['pos'], t['lemma'])

Con più ferie volereste più spesso?
volereste VERB volere
Se ci fosse meno vento, volereste anche oggi.
volereste VERB volere
A che ora volereste domani?
volereste VERB volere
Volereste davvero attraversare le Alpi di notte?
Volereste AUX volere
In quale compagnia volereste per risparmiare?
volereste VERB volere
Non volereste con un aereo così vecchio, vero?
volereste VERB volere
Mi confermate che volereste in economy?
volereste VERB volere
Se aveste il passaporto, volereste domani stesso.
volereste VERB volere
Volereste anche fare scalo a Roma?
Volereste AUX volere
Dove volereste con quei punti?
volereste VERB volere
In estate volereste quasi ogni weekend.
volereste VERB volere
Con quel budget volereste in economy.
volereste NOUN voleresta
